<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 05 — Compare, and Report

**Paired with L7.2 · Fundamental PDEs**

Two equations, four models, one lesson about how many conditions a problem
needs and what happens when it does not get them.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex07.2_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
files = {
    "01 · die, soft":     "nb01_die_soft.npz",
    "02 · die, hard":     "nb02_die_hard.npz",
    "03 · panel, both":   "nb03_panel.npz",
    "04 · panel, missing": "nb04_missing_ic.npz",
}
R = {}
cc.needed('nb01_die_soft.npz', 'nb02_die_hard.npz', 'nb03_panel.npz', 'nb04_missing_ic.npz')   # on Colab without Drive, asks for the missing files
for label, fn in files.items():
    p = os.path.join(cc.OUTPUT_DIR, fn)
    if os.path.exists(p):
        R[label] = np.load(p, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fn}) -- run that notebook first")

## 0b · Your personal seed

The notebooks fix the seed to 88 so the "what you should see" blocks are true
on any machine. The report asks for numbers from **your** seed instead, so a
report cannot be copied between groups without the numbers giving it away.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own collocation draw through the die's slab, and what the exact
# solution does on it.
your_pts = spacetime_points(3000, pb.HEAT_DOMAIN, (0.0, pb.HEAT_T_END), seed=SEED)
theta_you = pb.heat_exact(your_pts[:, 0], your_pts[:, 1], your_pts[:, 2])

your_panel = spacetime_points(3000, pb.WAVE_DOMAIN, (0.0, pb.WAVE_T_END), seed=SEED)
u_you = pb.wave_exact(your_panel[:, 0], your_panel[:, 1], your_panel[:, 2])

print()
print(f"  your mean die temperature   : {theta_you.mean():.5f} K")
print(f"  your peak die temperature   : {theta_you.max():.5f} K")
print(f"  your RMS panel displacement : {np.sqrt((u_you**2).mean())*1e6:.4f} um")

## 1 · The evidence, gathered

In [ ]:
if "01 · die, soft" in R and "02 · die, hard" in R:
    s, h = R["01 · die, soft"], R["02 · die, hard"]
    print("THE DIE — relative L2 against time")
    print(error_table(
        [[f"{t*1e3:.0f}", f"{a:.3e}", f"{b:.3e}",
          "-" if b == 0 else f"{a/b:.1f}x"]
         for t, a, b in zip(s["times"], s["rel"], h["rel"])],
        ["t [ms]", "soft IC", "hard IC", "ratio"]))
    print(f"\n  hard IC, error at t = 0 before training : "
          f"{float(h['ic_err']):.2e} K")

if "03 · panel, both" in R and "04 · panel, missing" in R:
    p, m = R["03 · panel, both"], R["04 · panel, missing"]
    print("\n\nTHE PANEL")
    print(f"  period, exact / measured  : {float(p['period_exact'])*1e3:.3f}"
          f" / {float(p['period_pred'])*1e3:.3f} ms")
    print(f"  worst error at the end    : {float(p['max_err'][-1])*1e6:.2f} um")
    print()
    print("  with the velocity condition omitted:")
    print(f"    fraction of real motion : {float(m['fraction'])*100:.3f} %")
    print(f"    final loss              : {float(m['final_loss']):.3e}")
    print(f"    final loss, correct run : {float(p['lbfgs'][-1]):.3e}")
    print(f"    PDE residual            : {float(m['pde_res']):.3e}   passes")
    print(f"    edge error              : {float(m['edge_err']):.3e} m  passes")
    print(f"    u(x,y,0) error          : {float(m['ic_disp']):.3e} m  passes")
    print(f"    u_t(x,y,0) error        : {float(m['ic_vel']):.4f} m/s  FAILS")

## 2 · Your answers

Replace every string. Keep to the word limits.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_COUNTING = """
(120 words) Give the rule for how many initial conditions a PDE requires, and
apply it to the die and the panel. Then state what physically distinguishes the
two -- not "one is first order" but what that means for the system.
"""

Q2_ERROR_IN_TIME = """
(150 words) The die's error fell with time; the panel's grew. Quote your own
numbers for both and give the mechanism. Say which of the two behaviours you
would expect for a convection-diffusion problem, and why.
"""

Q3_HARD_IC_COST = """
(150 words) The trial solution theta0 + t D N made the initial condition exact
before training. Report the untrained error, then describe what the raw network
N had to learn in exchange -- refer to the figure in notebook 02 section 5. Say
what you would expect if the time window were ten times longer.
"""

Q4_THE_TRIVIAL_SOLUTION = """
(150 words) The under-determined panel reached a LOWER loss than the correct
one. Explain why in terms of what u = 0 satisfies. Then quote the four
diagnostics from notebook 04 section 3 and say which of them you would have run
if you had not been told what to look for.
"""

Q5_NO_EXACT_SOLUTION = """
(150 words) From L8 onward you will usually have no exact solution. Given only
a residual and your own judgement, list three checks that could catch a mis-
posed problem, in the order you would run them. At least one must be something
you can do BEFORE training.
"""

Q6_WHAT_YOU_DISTRUST = """
(120 words) Name the result in this exercise you trust least, and the specific
experiment that would settle it.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

In [ ]:
answers = {
    "1 · Counting conditions": (Q1_COUNTING, 120),
    "2 · Where the error goes": (Q2_ERROR_IN_TIME, 150),
    "3 · What hard enforcement cost": (Q3_HARD_IC_COST, 150),
    "4 · The panel that never moved": (Q4_THE_TRIVIAL_SOLUTION, 150),
    "5 · When there is no exact solution": (Q5_NO_EXACT_SOLUTION, 150),
    "6 · What you distrust": (Q6_WHAT_YOU_DISTRUST, 120),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_07.2 — Fundamental PDEs: a die and a panel", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    out = os.path.join(cc.OUTPUT_DIR, "Ex07.2_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 04

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 17, each under its question, to the end of the
report you just wrote. The last one is not from a notebook: it is the
question across all of them, and it concludes the report. On Colab every
notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 04 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · The Die, with a **Soft** Initial Condition ----------------
    # 01.1 The error was largest at $t = 0$. Give the three reasons in your own
    # words and say which you think dominates here. Use them to explain why the
    # initial condition is the term a PINN most often loses, even though it is
    # the only place you were given the answer. (-> L7.2 Q7)
    "01.1": """
""",
    # 01.2 Both the edge term and the initial term were divided by the same
    # temperature scale, and the residual by a rate. Why does that matter, and
    # what would `w_0 = 1` have meant without it? Say whether raising `w_0`
    # alone would stop the initial slice being traded away. (-> L7.2 Q7)
    "01.2": """
""",
    # 01.3 Almost all the collocation points sit in the bulk of the slab and
    # are trained as one block. What does that do to the balance between the
    # three terms, and how would you change it? A time-marching solver cannot
    # get a late instant right before an early one; say what the PINN gains by
    # not marching and what this notebook shows it gives up. (-> L7.2 Q7)
    "01.3": """
""",
    # 01.4 The residual has one time derivative and two space derivatives. Say
    # which type of equation that makes the heat equation, what the type says
    # about the conditions it needs, and which loss term each of those
    # conditions became in this notebook. (-> L7.2 Q1, Q6)
    "01.4": """
""",

    # ---- notebook 02 · The Die, with the Initial Condition **Built In** ----------
    # 02.1 The untrained network already satisfied both conditions. Say
    # precisely why, term by term in $\theta_0 + t\,D\,N$. The lecture writes
    # the trial solution with a $(1 - t)$ factor on the initial field instead:
    # say what the two forms have in common and what the $(1 - t)$ form asks of
    # the time axis that this one does not. (-> L7.2 Q7)
    "02.1": """
""",
    # 02.2 The advantage over soft enforcement shrank with time. Explain the
    # mechanism, and relate it to how a PINN treats time: the whole block is
    # trained at once rather than stepped forward from the known field. (->
    # L7.2 Q7)
    "02.2": """
""",
    # 02.3 Look again at the raw network output $N$ at the centre. Say why it
    # must rise steeply as $t \to 0$ for the trial solution to decay correctly,
    # and what you would expect to go wrong if the time window were ten times
    # longer. (-> L7.2 Q7)
    "02.3": """
""",
    # 02.4 The initial field here is a formula. Describe what you would do if
    # it were a measured thermal image, and what new error you would be
    # introducing into the residual. Would you still build the initial
    # condition in, or keep it as a soft term? (-> L7.2 Q7)
    "02.4": """
""",

    # ---- notebook 03 · The Struck Panel ------------------------------------------
    # 03.1 The die's error fell with time and the panel's grew. Give the
    # mechanism, and tie it to the type of each equation: what does a parabolic
    # equation do to an error made early, and what does a hyperbolic one do?
    # (-> L7.2 Q1, Q6)
    "03.1": """
""",
    # 03.2 Why does a relative $L^2$ norm fail at $t = 0$ here but not in
    # notebook 01? Use the answer to say which of the panel's two initial
    # conditions actually carries the strike. (-> L7.2 Q9)
    "03.2": """
""",
    # 03.3 The period is a better first check than the pointwise error. Explain
    # what failure it catches that a norm would miss. Your tanh network tracked
    # the centre for a period or two and then drifted: what would a sine
    # activation buy on this problem, and how would you set up the comparison
    # so it measures the activation and not the network size? (-> L7.2 Q8)
    "03.3": """
""",
    # 03.4 You supplied both initial conditions. Before turning the page: why
    # does a second-order-in-time equation need two, and what do you think
    # happens if you supply only the displacement? (-> L7.2 Q9)
    "03.4": """
""",

    # ---- notebook 04 · What Happens If You Forget One ----------------------------
    # 04.1 State, in one sentence, why $u \equiv 0$ satisfied everything you
    # kept. The same trap returns for Helmholtz: say why the trivial solution
    # is admissible there but was never a risk for the slot's Poisson problem
    # in Ex_07.1. (-> L7.2 Q10, Q3)
    "04.1": """
""",
    # 04.2 The under-determined problem reached a **lower** loss than the
    # correct one. Explain why that is not surprising, and why it is dangerous:
    # what does a near-zero residual certify, and what does it not? (-> L7.2
    # Q10)
    "04.2": """
""",
    # 04.3 Give the counting rule for how many initial conditions a PDE needs,
    # and apply it to the die and to the panel. Say how the type of each
    # equation predicts the answer before any training. (-> L7.2 Q9, Q1)
    "04.3": """
""",
    # 04.4 Section 4 demonstrated one more way to be under-determined. Name a
    # steady-state one too: what goes wrong for Poisson with a prescribed flux
    # on every wall? Then list two checks you could still run on a problem with
    # no exact solution, such as L10's battery model, that would catch a
    # mis-posed problem. (-> L7.2 Q2, Q10)
    "04.4": """
""",

    # ---- to conclude, across all the notebooks ------------------------------
    # C Across the notebooks: the die's error fell with time and the panel's
    # grew, and the panel without its velocity condition reached the lowest
    # loss of all. What does the type of an equation tell you before training,
    # and what can a low loss not tell you? (-> L7.2, Exercise slide)
    "C": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'The Die, with a **Soft** Initial Condition', 'The error was largest at $t = 0$. Give the three reasons in your own words and say which you think dominates here. Use them to explain why the initial condition is the term a PINN most often loses, even though it is the only place you were given the answer.', 'L7.2 Q7'),
    "01.2": ('01', 'The Die, with a **Soft** Initial Condition', 'Both the edge term and the initial term were divided by the same temperature scale, and the residual by a rate. Why does that matter, and what would `w_0 = 1` have meant without it? Say whether raising `w_0` alone would stop the initial slice being traded away.', 'L7.2 Q7'),
    "01.3": ('01', 'The Die, with a **Soft** Initial Condition', 'Almost all the collocation points sit in the bulk of the slab and are trained as one block. What does that do to the balance between the three terms, and how would you change it? A time-marching solver cannot get a late instant right before an early one; say what the PINN gains by not marching and what this notebook shows it gives up.', 'L7.2 Q7'),
    "01.4": ('01', 'The Die, with a **Soft** Initial Condition', 'The residual has one time derivative and two space derivatives. Say which type of equation that makes the heat equation, what the type says about the conditions it needs, and which loss term each of those conditions became in this notebook.', 'L7.2 Q1, Q6'),
    "02.1": ('02', 'The Die, with the Initial Condition **Built In**', 'The untrained network already satisfied both conditions. Say precisely why, term by term in $\\theta_0 + t\\,D\\,N$. The lecture writes the trial solution with a $(1 - t)$ factor on the initial field instead: say what the two forms have in common and what the $(1 - t)$ form asks of the time axis that this one does not.', 'L7.2 Q7'),
    "02.2": ('02', 'The Die, with the Initial Condition **Built In**', 'The advantage over soft enforcement shrank with time. Explain the mechanism, and relate it to how a PINN treats time: the whole block is trained at once rather than stepped forward from the known field.', 'L7.2 Q7'),
    "02.3": ('02', 'The Die, with the Initial Condition **Built In**', 'Look again at the raw network output $N$ at the centre. Say why it must rise steeply as $t \\to 0$ for the trial solution to decay correctly, and what you would expect to go wrong if the time window were ten times longer.', 'L7.2 Q7'),
    "02.4": ('02', 'The Die, with the Initial Condition **Built In**', 'The initial field here is a formula. Describe what you would do if it were a measured thermal image, and what new error you would be introducing into the residual. Would you still build the initial condition in, or keep it as a soft term?', 'L7.2 Q7'),
    "03.1": ('03', 'The Struck Panel', "The die's error fell with time and the panel's grew. Give the mechanism, and tie it to the type of each equation: what does a parabolic equation do to an error made early, and what does a hyperbolic one do?", 'L7.2 Q1, Q6'),
    "03.2": ('03', 'The Struck Panel', "Why does a relative $L^2$ norm fail at $t = 0$ here but not in notebook 01? Use the answer to say which of the panel's two initial conditions actually carries the strike.", 'L7.2 Q9'),
    "03.3": ('03', 'The Struck Panel', 'The period is a better first check than the pointwise error. Explain what failure it catches that a norm would miss. Your tanh network tracked the centre for a period or two and then drifted: what would a sine activation buy on this problem, and how would you set up the comparison so it measures the activation and not the network size?', 'L7.2 Q8'),
    "03.4": ('03', 'The Struck Panel', 'You supplied both initial conditions. Before turning the page: why does a second-order-in-time equation need two, and what do you think happens if you supply only the displacement?', 'L7.2 Q9'),
    "04.1": ('04', 'What Happens If You Forget One', "State, in one sentence, why $u \\equiv 0$ satisfied everything you kept. The same trap returns for Helmholtz: say why the trivial solution is admissible there but was never a risk for the slot's Poisson problem in Ex_07.1.", 'L7.2 Q10, Q3'),
    "04.2": ('04', 'What Happens If You Forget One', 'The under-determined problem reached a **lower** loss than the correct one. Explain why that is not surprising, and why it is dangerous: what does a near-zero residual certify, and what does it not?', 'L7.2 Q10'),
    "04.3": ('04', 'What Happens If You Forget One', 'Give the counting rule for how many initial conditions a PDE needs, and apply it to the die and to the panel. Say how the type of each equation predicts the answer before any training.', 'L7.2 Q9, Q1'),
    "04.4": ('04', 'What Happens If You Forget One', "Section 4 demonstrated one more way to be under-determined. Name a steady-state one too: what goes wrong for Poisson with a prescribed flux on every wall? Then list two checks you could still run on a problem with no exact solution, such as L10's battery model, that would catch a mis-posed problem.", 'L7.2 Q2, Q10'),
    "C": ('C', 'to conclude', 'Across the notebooks: the die\'s error fell with time and the panel\'s grew, and the panel without its velocity condition reached the lowest loss of all. What does the type of an equation tell you before training, and what can a low loss not tell you?', 'L7.2, Exercise slide'),
}

report_md = os.path.join(cc.OUTPUT_DIR, "Ex07.2_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex07.2_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex07.2_report.md: 17 of 17 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.2_report.md into Ex07.2_report.pdf, with any figure
# saved as Ex07.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open(os.path.join(cc.OUTPUT_DIR, "Ex07.2_report.md"), encoding="utf-8").read()
figs = sorted(glob.glob("Ex07.2_report*.png")
              + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.2_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


## 4 · What Ex_07.2 was for

Four notebooks on a die and a panel, and three claims you can now defend with
your own measurements:

* **Where you put known information decides how much of it survives.** A
  condition in the loss is a request the optimiser may refuse; a condition in
  the function space is a fact it cannot.
* **Hard enforcement is not free.** It needs a geometry you can write down, it
  hard-codes the boundary *value*, it does not extend to conditions on
  derivatives, and it hands the network a harder function to learn.
* **A small residual is not a correct answer.** The panel that never moved had
  the best loss in the whole exercise set.

From **L8** the equations acquire real geometry and, more importantly, real
consequences: a plate with a hole, a channel, a battery, a grid. The machinery
stops changing. What changes is that you will less and less often have an exact
solution to check against — so the habits built here, counting conditions and
distrusting a good loss, are the ones that carry.